___
<h3> Read PDFs from the data folder, and uses pymudpdf4ll to convert pages to the markdown. With extracted markdown, we perform table extraction and create data structure later to be stored and reused for further processing </h3>

___ 

In [ ]:
import pymupdf4llm
import os
import json
import re

def split_md_row(row: str):
    row = row.strip()
    if not row.startswith("|") or "|" not in row[1:]:
        return []
    cells = [c.strip() for c in row.strip("|").split("|")]
    while cells and cells[-1] == "":
        cells.pop()
    return cells

def is_separator_row(cells):
    if not cells:
        return False
    return all(re.fullmatch(r":?-{3,}:?", c.replace(" ", "")) for c in cells if c != "")

def normalize_header(h: str, idx: int):
    h = re.sub(r"\s+", "_", h.strip().lower())
    h = re.sub(r"[^a-z0-9_]+", "", h)
    return h if h else f"col_{idx+1}"

def parse_tables_from_markdown(md_text: str):
    lines = (md_text or "").splitlines()
    tables = []
    i = 0

    while i < len(lines) - 1:
        h_cells = split_md_row(lines[i])
        s_cells = split_md_row(lines[i + 1])

        if h_cells and s_cells and len(h_cells) == len(s_cells) and is_separator_row(s_cells):
            headers = [normalize_header(h, idx) for idx, h in enumerate(h_cells)]
            rows = []
            i += 2

            while i < len(lines):
                r_cells = split_md_row(lines[i])
                if not r_cells:
                    break

                if len(r_cells) < len(headers):
                    r_cells = r_cells + [""] * (len(headers) - len(r_cells))
                elif len(r_cells) > len(headers):
                    r_cells = r_cells[:len(headers)]

                rows.append({headers[c]: r_cells[c] for c in range(len(headers))})
                i += 1

            tables.append({
                "headers": headers,
                "rows": rows
            })
        else:
            i += 1

    return tables

data_folder = "data"
output_path = "data_processed/pages_with_tables.json"

pdf_files = [f for f in os.listdir(data_folder) if f.lower().endswith(".pdf")]
documents = []

for pdf_file in pdf_files:
    pdf_path = os.path.join(data_folder, pdf_file)
    page_chunks = pymupdf4llm.to_markdown(pdf_path, page_chunks=True)

    pages = []
    for idx, chunk in enumerate(page_chunks, start=1):
        if isinstance(chunk, dict):
            page_num = chunk.get("page", chunk.get("number", idx))
            page_md = chunk.get("text") or chunk.get("md") or chunk.get("content") or ""
        else:
            page_num = idx
            page_md = str(chunk)

        tables = parse_tables_from_markdown(page_md)

        pages.append({
            "page_num": page_num,
            "text": page_md,
            "tables": tables
        })

    documents.append({
        "document_name": pdf_file,
        "pages": pages
    })

In [ ]:
documents[0] if documents else {}

___
<h3> Each document is targeting some set of products, as documents typically reflect specs for the product series. With this we extract the exact product series and note it down </h3>

___

In [ ]:
TARGET_COLUMN = "commercial_product_name"

for doc in documents:
    product_names = []
    for page in doc.get("pages", []):
        for table in page.get("tables", []):
            if TARGET_COLUMN in table.get("headers", []):
                for row in table.get("rows", []):
                    val = row.get(TARGET_COLUMN, "").strip()
                    if val and val not in product_names:
                        product_names.append(val)
    doc["targeted_products"] = ", ".join(product_names)

___
<h3> By manually inspecting the tables, not all tables are related to all product series. Sometimes tables are split into categories per product serie. Which product series are targeted is typically noted down before the table. So the cell below does the brute force search, and checks if the last N number of lines before the cell contains any product names (product names are placed in the targeted_products by the previous cell) </h3>

___

In [ ]:
CONTEXT_LINES = 20  # lines before table start to inspect for product names

def find_products_in_context(context_text: str, targeted_products: list) -> list:
    """Match targeted_products against pre-table context using exact and space-stripped matching."""
    if not targeted_products:
        return []
    context_lower = context_text.lower()
    context_nospace = context_lower.replace(" ", "")
    found = []
    for product in targeted_products:
        p_lower = product.lower()
        p_nospace = p_lower.replace(" ", "")
        if p_lower in context_lower or (p_nospace and p_nospace in context_nospace):
            found.append(product)
    return found


def get_pre_table_contexts(page_text: str) -> list:
    """
    Re-scan page markdown and return list of pre-table context strings,
    one per table (in table order). Reuses split_md_row / is_separator_row.
    """
    lines = (page_text or "").splitlines()
    contexts = []
    i = 0
    while i < len(lines) - 1:
        h_cells = split_md_row(lines[i])
        s_cells = split_md_row(lines[i + 1])
        if h_cells and s_cells and len(h_cells) == len(s_cells) and is_separator_row(s_cells):
            context_start = max(0, i - CONTEXT_LINES)
            context_lines = [l for l in lines[context_start:i] if l.strip()]
            contexts.append("\n".join(context_lines))
            # skip past the table body
            i += 2
            while i < len(lines) and split_md_row(lines[i]):
                i += 1
        else:
            i += 1
    return contexts


for doc in documents:
    targeted_products_str = doc.get("targeted_products", "")
    targeted_products = [p.strip() for p in targeted_products_str.split(",") if p.strip()]

    for page in doc.get("pages", []):
        tables = page.get("tables", [])
        if not tables:
            continue

        pre_table_contexts = get_pre_table_contexts(page.get("text", ""))

        for t_idx, table in enumerate(tables):
            context = pre_table_contexts[t_idx] if t_idx < len(pre_table_contexts) else ""
            related = find_products_in_context(context, targeted_products)

            # Fallback: relate to all document products if nothing specific found
            if not related:
                related = targeted_products

            related_str = ", ".join(related)

            # Add related_products column if not present (and if commercial_product_name isn't already there to avoid redundancy)
            if "related_products" not in table["headers"] and 'commercial_product_name' not in table["headers"]:
                table["headers"].append("related_products")
            for row in table.get("rows", []):
                row["related_products"] = related_str

In [ ]:
documents[0]['pages'] if documents and documents[0].get("pages") else {}


In [ ]:
os.makedirs("data_processed", exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)